In [13]:
# ============================================================
# RELATIVE DENSITY MINI-MODEL
# Input  : P, v, h, t
# Output : Relative Density (%)
#
# Experimental dataset: 20 points
# ============================================================

In [14]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


In [15]:
# ------------------------------------------------------------
# SKLEARN
# ------------------------------------------------------------

from sklearn.model_selection import LeaveOneOut, GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [16]:
# ------------------------------------------------------------
# OPTIONAL MODELS
# ------------------------------------------------------------

models_optional = {}

try:
    from xgboost import XGBRegressor

    models_optional["XGBoost"] = XGBRegressor(
        n_estimators=200,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42
    )

except ImportError:
    print("XGBoost not installed -- skipping XGBoost.")


try:
    from catboost import CatBoostRegressor

    models_optional["CatBoost"] = CatBoostRegressor(
        iterations=200,
        depth=3,
        learning_rate=0.03,
        l2_leaf_reg=5,
        loss_function="RMSE",
        verbose=False,
        random_seed=42
    )

except ImportError:
    print("CatBoost not installed -- skipping CatBoost.")

In [17]:
# ============================================================
# EXPERIMENTAL DATA — UNIQUE PROCESS CONDITIONS ONLY
# ============================================================

import pandas as pd

data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 260, 290, 280, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1600, 1450,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1500, 1550, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.075, 0.08, 0.075, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "RelativeDensity_pct": [
        99.30, 99.80, 99.80, 99.90, 98.20,
        99.20, 99.10, 99.10, 99.50, 99.90,
        99.95, 99.20, 99.70, 98.80, 99.95
    ]
})

print("Number of unique process conditions:", len(data))
print(data)

Number of unique process conditions: 15
    P_W  v_mm_per_s   h_mm  t_mm  RelativeDensity_pct
0   260        1550  0.075  0.04                99.30
1   280        1450  0.075  0.04                99.80
2   270        1500  0.090  0.04                99.80
3   270        1600  0.080  0.04                99.90
4   270        1450  0.080  0.04                98.20
5   260        1450  0.085  0.04                99.20
6   270        1500  0.070  0.04                99.10
7   250        1500  0.080  0.04                99.10
8   260        1550  0.085  0.04                99.50
9   270        1400  0.080  0.04                99.90
10  280        1550  0.085  0.04                99.95
11  260        1450  0.075  0.04                99.20
12  290        1500  0.080  0.04                99.70
13  280        1550  0.075  0.04                98.80
14  270        1500  0.080  0.04                99.95


In [18]:
# ============================================================
# 2. INPUT / TARGET
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

target = "RelativeDensity_pct"

X = data[features]
y = data[target]

In [19]:
# ============================================================
# 3. SHOW DATA
# ============================================================

print("=" * 70)
print("RELATIVE DENSITY MINI-MODEL")
print("=" * 70)

print("\nExperimental data:")
print(data.to_string(index=False))

print("\nTarget statistics:")
print(y.describe())

RELATIVE DENSITY MINI-MODEL

Experimental data:
 P_W  v_mm_per_s  h_mm  t_mm  RelativeDensity_pct
 260        1550 0.075  0.04                99.30
 280        1450 0.075  0.04                99.80
 270        1500 0.090  0.04                99.80
 270        1600 0.080  0.04                99.90
 270        1450 0.080  0.04                98.20
 260        1450 0.085  0.04                99.20
 270        1500 0.070  0.04                99.10
 250        1500 0.080  0.04                99.10
 260        1550 0.085  0.04                99.50
 270        1400 0.080  0.04                99.90
 280        1550 0.085  0.04                99.95
 260        1450 0.075  0.04                99.20
 290        1500 0.080  0.04                99.70
 280        1550 0.075  0.04                98.80
 270        1500 0.080  0.04                99.95

Target statistics:
count    15.000000
mean     99.426667
std       0.505988
min      98.200000
25%      99.150000
50%      99.500000
75%      99.850000

In [20]:
# ============================================================
# 4. DEFINE MODELS
# ============================================================

models = {

    # --------------------------------------------------------
    # Linear
    # --------------------------------------------------------
    "Linear Regression":
        LinearRegression(),

    # --------------------------------------------------------
    # Polynomial Degree 2
    # --------------------------------------------------------
    "Polynomial Degree 2":
        Pipeline([
            ("poly", PolynomialFeatures(
                degree=2,
                include_bias=False
            )),
            ("model", LinearRegression())
        ]),

    # --------------------------------------------------------
    # Polynomial Degree 3
    # --------------------------------------------------------
    "Polynomial Degree 3":
        Pipeline([
            ("poly", PolynomialFeatures(
                degree=3,
                include_bias=False
            )),
            ("model", Ridge(alpha=1.0))
        ]),

    # --------------------------------------------------------
    # Ridge
    # --------------------------------------------------------
    "Ridge":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),

    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------
    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            max_depth=4,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),

    # --------------------------------------------------------
    # Extra Trees
    # --------------------------------------------------------
    "Extra Trees":
        ExtraTreesRegressor(
            n_estimators=300,
            max_depth=4,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),

    # --------------------------------------------------------
    # Gradient Boosting
    # --------------------------------------------------------
    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.03,
            min_samples_leaf=2,
            random_state=42
        )
}


# Add optional models
models.update(models_optional)

In [21]:
# ============================================================
# 5. LOOCV
# ============================================================

loo = LeaveOneOut()

results = []
predictions = {}

for name, model in models.items():

    y_pred = cross_val_predict(
        model,
        X,
        y,
        cv=loo
    )

    rmse = np.sqrt(
        mean_squared_error(y, y_pred)
    )

    mae = mean_absolute_error(
        y, y_pred
    )

    r2 = r2_score(
        y, y_pred
    )

    # Fit on all data for training R2
    model.fit(X, y)

    y_train_pred = model.predict(X)

    train_r2 = r2_score(
        y,
        y_train_pred
    )

    results.append({
        "Model": name,
        "LOOCV_RMSE_pct": rmse,
        "LOOCV_MAE_pct": mae,
        "LOOCV_R2": r2,
        "Training_R2": train_r2
    })

    predictions[name] = y_pred

In [22]:
# ============================================================
# 6. RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="LOOCV_RMSE_pct"
).reset_index(drop=True)


print("\n")
print("=" * 70)
print("LOOCV MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)



LOOCV MODEL COMPARISON
              Model  LOOCV_RMSE_pct  LOOCV_MAE_pct  LOOCV_R2  Training_R2
              Ridge          0.5397         0.3800   -0.2190       0.2035
  Linear Regression          0.5451         0.3833   -0.2436       0.2043
        Extra Trees          0.5725         0.4467   -0.3718       0.2381
      Random Forest          0.5911         0.4302   -0.4621       0.3115
           CatBoost          0.6597         0.4971   -0.8210       0.5240
            XGBoost          0.7222         0.5428   -1.1828       0.5418
  Gradient Boosting          0.7368         0.5478   -1.2721       0.4391
Polynomial Degree 2          0.7470         0.4742   -1.3355       0.4404
Polynomial Degree 3          0.7933         0.6531   -1.6336       0.6868


In [23]:
# ============================================================
# 7. BEST MODEL
# ============================================================

best_name = results_df.iloc[0]["Model"]

best_model = models[best_name]

best_model.fit(X, y)

best_pred = predictions[best_name]


print("\n")
print("=" * 70)
print("BEST MODEL")
print("=" * 70)

print("Model:", best_name)

print(
    f"LOOCV RMSE: "
    f"{results_df.iloc[0]['LOOCV_RMSE_pct']:.4f} %"
)

print(
    f"LOOCV MAE : "
    f"{results_df.iloc[0]['LOOCV_MAE_pct']:.4f} %"
)

print(
    f"LOOCV R²  : "
    f"{results_df.iloc[0]['LOOCV_R2']:.4f}"
)



BEST MODEL
Model: Ridge
LOOCV RMSE: 0.5397 %
LOOCV MAE : 0.3800 %
LOOCV R²  : -0.2190


In [24]:
# ============================================================
# 8. ACTUAL VS LOOCV PREDICTION
# ============================================================

comparison = data.copy()

comparison["Predicted_Density_pct"] = best_pred

comparison["Error_pct"] = (
    comparison["Predicted_Density_pct"]
    - comparison["RelativeDensity_pct"]
)

comparison["Absolute_Error_pct"] = (
    comparison["Error_pct"].abs()
)


print("\n")
print("=" * 70)
print("ACTUAL VS LOOCV PREDICTION")
print("=" * 70)

print(
    comparison[
        [
            "P_W",
            "v_mm_per_s",
            "h_mm",
            "t_mm",
            "RelativeDensity_pct",
            "Predicted_Density_pct",
            "Error_pct"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)



ACTUAL VS LOOCV PREDICTION
 P_W  v_mm_per_s  h_mm  t_mm  RelativeDensity_pct  Predicted_Density_pct  Error_pct
 260        1550 0.075 0.040               99.300                 99.179     -0.121
 280        1450 0.075 0.040               99.800                 99.233     -0.567
 270        1500 0.090 0.040               99.800                 99.740     -0.060
 270        1600 0.080 0.040               99.900                 99.394     -0.506
 270        1450 0.080 0.040               98.200                 99.569      1.369
 260        1450 0.085 0.040               99.200                 99.490      0.290
 270        1500 0.070 0.040               99.100                 99.146      0.046
 250        1500 0.080 0.040               99.100                 99.215      0.115
 260        1550 0.085 0.040               99.500                 99.527      0.027
 270        1400 0.080 0.040               99.900                 99.094     -0.806
 280        1550 0.085 0.040               99.9

In [25]:
# ============================================================
# 9. GROUPED VALIDATION
#
# Same process condition occurs multiple times.
# We create a group based on P, v and h.
# t is constant in this dataset.
# ============================================================

data["ProcessGroup"] = (
    data["P_W"].astype(str)
    + "_"
    + data["v_mm_per_s"].astype(str)
    + "_"
    + data["h_mm"].astype(str)
    + "_"
    + data["t_mm"].astype(str)
)

groups = data["ProcessGroup"]

n_groups = groups.nunique()

print("\n")
print("=" * 70)
print("PROCESS CONDITION INFORMATION")
print("=" * 70)

print("Total observations:", len(data))
print("Unique process conditions:", n_groups)

print("\nRepeated conditions:")
print(
    data["ProcessGroup"]
    .value_counts()
    .loc[lambda x: x > 1]
)



PROCESS CONDITION INFORMATION
Total observations: 15
Unique process conditions: 15

Repeated conditions:
Series([], Name: count, dtype: int64)


In [26]:
# ============================================================
# GROUP K-FOLD
# ============================================================

if n_groups >= 5:

    n_splits = min(5, n_groups)

    gkf = GroupKFold(
        n_splits=n_splits
    )

    grouped_results = []

    for name, model in models.items():

        y_group_pred = cross_val_predict(
            model,
            X,
            y,
            cv=gkf,
            groups=groups
        )

        rmse = np.sqrt(
            mean_squared_error(
                y,
                y_group_pred
            )
        )

        mae = mean_absolute_error(
            y,
            y_group_pred
        )

        r2 = r2_score(
            y,
            y_group_pred
        )

        grouped_results.append({
            "Model": name,
            "Grouped_RMSE_pct": rmse,
            "Grouped_MAE_pct": mae,
            "Grouped_R2": r2
        })

    grouped_df = pd.DataFrame(
        grouped_results
    ).sort_values(
        "Grouped_RMSE_pct"
    )

    print("\n")
    print("=" * 70)
    print("GROUPED CROSS-VALIDATION")
    print("=" * 70)

    print(
        grouped_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )



GROUPED CROSS-VALIDATION
              Model  Grouped_RMSE_pct  Grouped_MAE_pct  Grouped_R2
              Ridge            0.5104           0.3478     -0.0903
  Linear Regression            0.5129           0.3453     -0.1011
      Random Forest            0.5868           0.4287     -0.4411
        Extra Trees            0.5870           0.4671     -0.4422
           CatBoost            0.6665           0.5130     -0.8591
            XGBoost            0.7152           0.5445     -1.1406
  Gradient Boosting            0.7294           0.5685     -1.2264
Polynomial Degree 3            0.8605           0.6823     -2.0989
Polynomial Degree 2            1.2196           0.9882     -5.2250


In [27]:
# ============================================================
# 10. TRAIN BEST MODEL ON ALL EXPERIMENTAL DATA
# ============================================================

best_model.fit(X, y)


,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None


In [28]:
# ============================================================
# 11. EXAMPLE PREDICTION
# ============================================================

example = pd.DataFrame({
    "P_W": [270],
    "v_mm_per_s": [1500],
    "h_mm": [0.08],
    "t_mm": [0.04]
})

example_prediction = best_model.predict(example)[0]

print("\n")
print("=" * 70)
print("EXAMPLE PREDICTION")
print("=" * 70)

print(
    "Input:"
)

print(example)

print(
    f"\nPredicted Relative Density = "
    f"{example_prediction:.3f} %"
)




EXAMPLE PREDICTION
Input:
   P_W  v_mm_per_s  h_mm  t_mm
0  270        1500  0.08  0.04

Predicted Relative Density = 99.446 %


In [ ]:
# ============================================================
# 12. SAVE BEST MODEL
# ============================================================

import joblib

joblib.dump(
    best_model,
    "RelativeDensity_mini_model.pkl"
)

joblib.dump(
    features,
    "RelativeDensity_feature_order.pkl"
)

print("\n")
print("Best Relative Density mini-model saved:")
print("RelativeDensity_mini_model.pkl")

In [30]:
# ============================================================
# RELATIVE DENSITY MINI-MODEL
#
# INPUT:
#   P, v, h, t
#
# OUTPUT:
#   Relative Density (%)
#
# Dataset:
#   20 experimental measurements
#   repeated process conditions averaged
#
# Validation:
#   Leave-One-Process-Condition-Out
# ============================================================

import numpy as np
import pandas as pd
import warnings
import joblib

warnings.filterwarnings("ignore")

# ============================================================
# 1. EXPERIMENTAL DATA
# ============================================================

raw_data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "RelativeDensity_pct": [
        99.30, 99.80, 99.80, 99.90, 98.20,
        99.20, 99.10, 99.10, 99.50, 99.90,
        99.95, 99.70, 99.20, 99.70, 98.80,
        99.70, 99.95, 99.40, 99.10, 99.80
    ]
})


# ============================================================
# 2. AVERAGE REPEATED PROCESS CONDITIONS
# ============================================================

group_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    raw_data
    .groupby(group_cols, as_index=False)
    ["RelativeDensity_pct"]
    .mean()
)

print("=" * 70)
print("UNIQUE PROCESS-CONDITION DATASET")
print("=" * 70)

print(data.to_string(index=False))

print("\nNumber of original measurements :", len(raw_data))
print("Number of unique conditions     :", len(data))


# ============================================================
# 3. INPUT / TARGET
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

target = "RelativeDensity_pct"

X = data[features]
y = data[target]


# ============================================================
# 4. IMPORT ML MODELS
# ============================================================

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 5. DEFINE CANDIDATE MODELS
# ============================================================

models = {

    # --------------------------------------------------------
    # MODEL 1: Linear Regression
    # --------------------------------------------------------
    "Linear Regression":
        LinearRegression(),


    # --------------------------------------------------------
    # MODEL 2: Polynomial Degree 2
    # --------------------------------------------------------
    "Polynomial Degree 2":
        Pipeline([
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False
                )
            ),
            (
                "model",
                LinearRegression()
            )
        ]),


    # --------------------------------------------------------
    # MODEL 3: Polynomial Degree 3 + Ridge
    # --------------------------------------------------------
    "Polynomial Degree 3 + Ridge":
        Pipeline([
            (
                "poly",
                PolynomialFeatures(
                    degree=3,
                    include_bias=False
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                Ridge(alpha=1.0)
            )
        ]),


    # --------------------------------------------------------
    # MODEL 4: Ridge
    # --------------------------------------------------------
    "Ridge":
        Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                Ridge(alpha=1.0)
            )
        ]),


    # --------------------------------------------------------
    # MODEL 5: Lasso
    # --------------------------------------------------------
    "Lasso":
        Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                Lasso(alpha=0.01)
            )
        ]),


    # --------------------------------------------------------
    # MODEL 6: Random Forest
    # --------------------------------------------------------
    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),


    # --------------------------------------------------------
    # MODEL 7: Extra Trees
    # --------------------------------------------------------
    "Extra Trees":
        ExtraTreesRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),


    # --------------------------------------------------------
    # MODEL 8: Gradient Boosting
    # --------------------------------------------------------
    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.03,
            min_samples_leaf=2,
            random_state=42
        )
}


# ============================================================
# 6. OPTIONAL XGBOOST
# ============================================================

try:

    from xgboost import XGBRegressor

    models["XGBoost"] = XGBRegressor(
        n_estimators=200,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=42
    )

    print("\nXGBoost available.")

except ImportError:

    print("\nXGBoost not installed.")


# ============================================================
# 7. OPTIONAL CATBOOST
# ============================================================

try:

    from catboost import CatBoostRegressor

    models["CatBoost"] = CatBoostRegressor(
        iterations=200,
        depth=3,
        learning_rate=0.03,
        l2_leaf_reg=5,
        loss_function="RMSE",
        verbose=False,
        random_seed=42
    )

    print("CatBoost available.")

except ImportError:

    print("CatBoost not installed.")


# ============================================================
# 8. LOOCV
# ============================================================

loo = LeaveOneOut()

results = []
all_predictions = {}


for name, model in models.items():

    # --------------------------------------------------------
    # Leave-One-Out predictions
    # --------------------------------------------------------

    y_pred = cross_val_predict(
        model,
        X,
        y,
        cv=loo
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    rmse = np.sqrt(
        mean_squared_error(y, y_pred)
    )

    mae = mean_absolute_error(
        y,
        y_pred
    )

    r2 = r2_score(
        y,
        y_pred
    )

    # --------------------------------------------------------
    # Training R²
    # --------------------------------------------------------

    model.fit(X, y)

    y_train_pred = model.predict(X)

    train_r2 = r2_score(
        y,
        y_train_pred
    )

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    results.append({

        "Model": name,

        "LOOCV_RMSE_pct":
            rmse,

        "LOOCV_MAE_pct":
            mae,

        "LOOCV_R2":
            r2,

        "Training_R2":
            train_r2
    })

    all_predictions[name] = y_pred


# ============================================================
# 9. MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)

results_df = (
    results_df
    .sort_values(
        by="LOOCV_RMSE_pct"
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 70)
print("RELATIVE DENSITY MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 10. BEST MODEL
# ============================================================

best_name = results_df.iloc[0]["Model"]

best_model = models[best_name]

best_model.fit(X, y)

best_predictions = all_predictions[best_name]


print("\n")
print("=" * 70)
print("BEST MODEL")
print("=" * 70)

print("Selected model:", best_name)

print(
    f"LOOCV RMSE = "
    f"{results_df.iloc[0]['LOOCV_RMSE_pct']:.4f} %"
)

print(
    f"LOOCV MAE  = "
    f"{results_df.iloc[0]['LOOCV_MAE_pct']:.4f} %"
)

print(
    f"LOOCV R²   = "
    f"{results_df.iloc[0]['LOOCV_R2']:.4f}"
)

print(
    f"Training R² = "
    f"{results_df.iloc[0]['Training_R2']:.4f}"
)


# ============================================================
# 11. ACTUAL VS PREDICTED
# ============================================================

comparison = data.copy()

comparison["Predicted_Density_pct"] = (
    best_predictions
)

comparison["Error_pct"] = (
    comparison["Predicted_Density_pct"]
    - comparison["RelativeDensity_pct"]
)

comparison["Absolute_Error_pct"] = (
    comparison["Error_pct"].abs()
)


print("\n")
print("=" * 70)
print("ACTUAL VS LOOCV PREDICTED")
print("=" * 70)

print(
    comparison[
        [
            "P_W",
            "v_mm_per_s",
            "h_mm",
            "t_mm",
            "RelativeDensity_pct",
            "Predicted_Density_pct",
            "Error_pct"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 12. FIT BEST MODEL USING ALL 15 CONDITIONS
# ============================================================

best_model.fit(X, y)


# ============================================================
# 13. EXAMPLE PREDICTION
# ============================================================

example = pd.DataFrame({

    "P_W": [270],

    "v_mm_per_s": [1500],

    "h_mm": [0.08],

    "t_mm": [0.04]
})

predicted_density = (
    best_model.predict(example)[0]
)

print("\n")
print("=" * 70)
print("EXAMPLE PREDICTION")
print("=" * 70)

print(example)

print(
    f"\nPredicted Relative Density = "
    f"{predicted_density:.3f} %"
)


# ============================================================
# 14. SAVE BEST MINI-MODEL
# ============================================================

joblib.dump(
    best_model,
    "RelativeDensity_mini_model.pkl"
)

joblib.dump(
    features,
    "RelativeDensity_feature_order.pkl"
)

print("\n")
print("Saved:")
print("  RelativeDensity_mini_model.pkl")
print("  RelativeDensity_feature_order.pkl")

UNIQUE PROCESS-CONDITION DATASET
 P_W  v_mm_per_s  h_mm  t_mm  RelativeDensity_pct
 250        1500 0.080  0.04            99.100000
 260        1450 0.075  0.04            99.200000
 260        1450 0.085  0.04            99.200000
 260        1550 0.075  0.04            99.300000
 260        1550 0.085  0.04            99.500000
 270        1400 0.080  0.04            99.900000
 270        1500 0.070  0.04            99.100000
 270        1500 0.080  0.04            99.641667
 270        1500 0.090  0.04            99.800000
 270        1600 0.080  0.04            98.200000
 280        1450 0.075  0.04            99.800000
 280        1450 0.085  0.04            99.700000
 280        1550 0.075  0.04            98.800000
 280        1550 0.085  0.04            99.950000
 290        1500 0.080  0.04            99.700000

Number of original measurements : 20
Number of unique conditions     : 15

XGBoost available.
CatBoost available.


RELATIVE DENSITY MODEL COMPARISON
                

In [31]:
# ============================================================
# PHYSICS-INSPIRED FEATURE TEST
# ============================================================

data["VED_J_per_mm3"] = (
    data["P_W"]
    / (
        data["v_mm_per_s"]
        * data["h_mm"]
        * data["t_mm"]
    )
)

data["ED_J_per_mm2"] = (
    data["P_W"]
    / (
        data["v_mm_per_s"]
        * data["h_mm"]
    )
)

data["log_VED"] = np.log(
    data["VED_J_per_mm3"]
)


feature_sets = {

    "P_v_h_t": [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm"
    ],

    "VED_only": [
        "VED_J_per_mm3"
    ],

    "P_v_h_t_VED": [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm",
        "VED_J_per_mm3"
    ],

    "P_v_h_t_logVED": [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm",
        "log_VED"
    ],

    "VED_ED": [
        "VED_J_per_mm3",
        "ED_J_per_mm2"
    ]
}


feature_results = []


for feature_name, feature_list in feature_sets.items():

    X_test = data[feature_list]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    y_pred = cross_val_predict(
        model,
        X_test,
        y,
        cv=LeaveOneOut()
    )

    rmse = np.sqrt(
        mean_squared_error(y, y_pred)
    )

    mae = mean_absolute_error(
        y, y_pred
    )

    r2 = r2_score(
        y, y_pred
    )

    feature_results.append({

        "Feature_Set": feature_name,
        "LOOCV_RMSE_pct": rmse,
        "LOOCV_MAE_pct": mae,
        "LOOCV_R2": r2
    })


feature_results_df = pd.DataFrame(
    feature_results
).sort_values(
    "LOOCV_RMSE_pct"
)

print("\n")
print("=" * 70)
print("PHYSICS-INSPIRED FEATURE COMPARISON")
print("=" * 70)

print(
    feature_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)



PHYSICS-INSPIRED FEATURE COMPARISON
   Feature_Set  LOOCV_RMSE_pct  LOOCV_MAE_pct  LOOCV_R2
       P_v_h_t          0.4360         0.3351    0.1060
   P_v_h_t_VED          0.4367         0.3342    0.1031
P_v_h_t_logVED          0.4368         0.3353    0.1029
      VED_only          0.5252         0.4395   -0.2970
        VED_ED          0.5264         0.4406   -0.3028


In [1]:
# ============================================================
# RELATIVE DENSITY MINI-MODEL — IMPROVED VERSION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. EXPERIMENTAL DATA
# ============================================================

data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "RelativeDensity_pct": [
        99.30, 99.80, 99.80, 99.90, 98.20,
        99.20, 99.10, 99.10, 99.50, 99.90,
        99.95, 99.70, 99.20, 99.70, 98.80,
        99.70, 99.95, 99.40, 99.10, 99.80
    ]
})


# ============================================================
# 2. REMOVE DUPLICATE PROCESS CONDITIONS
# ============================================================

process_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    data
    .groupby(process_cols, as_index=False)
    ["RelativeDensity_pct"]
    .mean()
)

print("Unique experimental conditions:", len(data))


# ============================================================
# 3. PHYSICS FEATURES
# ============================================================

Cp = 436.0
k_mm = 0.0103

data["VED"] = (
    data["P_W"] /
    (
        data["v_mm_per_s"]
        * data["h_mm"]
        * data["t_mm"]
    )
)

data["Pi1"] = (
    Cp * data["P_W"]
    /
    (
        k_mm
        * data["v_mm_per_s"]**2
        * data["h_mm"]
    )
)


# ============================================================
# 4. CREATE DENSITY DEFICIT
# ============================================================

density = data["RelativeDensity_pct"] / 100.0

density_deficit = 1.0 - density

data["DensityDeficit"] = density_deficit

data["LogDensityDeficit"] = np.log(
    density_deficit
)


# ============================================================
# 5. FEATURES
# ============================================================

X_process = data[
    [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm"
    ]
]

X_pi1 = data[
    [
        "Pi1"
    ]
]

y_deficit = data[
    "DensityDeficit"
].values

y_log_deficit = data[
    "LogDensityDeficit"
].values


# ============================================================
# 6. EVALUATION FUNCTION
# ============================================================

loo = LeaveOneOut()

results = []


def evaluate(
    name,
    model,
    X,
    y,
    transform=False
):

    prediction = cross_val_predict(
        model,
        X,
        y,
        cv=loo
    )

    if transform:

        deficit_pred = np.exp(
            prediction
        )

    else:

        deficit_pred = prediction


    # Physical constraint
    deficit_pred = np.clip(
        deficit_pred,
        0,
        1
    )

    density_pred = (
        1.0 - deficit_pred
    ) * 100.0

    density_actual = (
        1.0 - y_deficit
    ) * 100.0


    rmse = np.sqrt(
        mean_squared_error(
            density_actual,
            density_pred
        )
    )

    mae = mean_absolute_error(
        density_actual,
        density_pred
    )

    r2 = r2_score(
        density_actual,
        density_pred
    )


    results.append({

        "Model": name,

        "LOOCV_RMSE_pct":
            rmse,

        "LOOCV_MAE_pct":
            mae,

        "LOOCV_R2":
            r2
    })


# ============================================================
# 7. RIDGE — DENSITY DEFICIT
# ============================================================

evaluate(

    "Ridge - Density Deficit",

    Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=10)
        )
    ]),

    X_process,

    y_deficit
)


# ============================================================
# 8. RIDGE — LOG DENSITY DEFICIT
# ============================================================

evaluate(

    "Ridge - Log Density Deficit",

    Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=10)
        )
    ]),

    X_process,

    y_log_deficit,

    transform=True
)


# ============================================================
# 9. RIDGE — Pi1 ONLY
# ============================================================

evaluate(

    "Ridge - Pi1",

    Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=10)
        )
    ]),

    X_pi1,

    y_deficit
)


# ============================================================
# 10. EXTRA TREES
# ============================================================

evaluate(

    "Extra Trees - Density Deficit",

    ExtraTreesRegressor(
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42
    ),

    X_process,

    y_deficit
)


# ============================================================
# 11. RANDOM FOREST
# ============================================================

evaluate(

    "Random Forest - Density Deficit",

    RandomForestRegressor(
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42
    ),

    X_process,

    y_deficit
)


# ============================================================
# 12. GRADIENT BOOSTING
# ============================================================

evaluate(

    "Gradient Boosting - Density Deficit",

    GradientBoostingRegressor(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.03,
        loss="huber",
        random_state=42
    ),

    X_process,

    y_deficit
)


# ============================================================
# 13. RESULTS
# ============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values(
        "LOOCV_RMSE_pct"
    )
    .reset_index(drop=True)
)

print("\n")
print("=" * 70)
print("IMPROVED RELATIVE DENSITY MINI-MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

Unique experimental conditions: 15


IMPROVED RELATIVE DENSITY MINI-MODEL COMPARISON
                              Model  LOOCV_RMSE_pct  LOOCV_MAE_pct  LOOCV_R2
            Ridge - Density Deficit          0.4278         0.3344    0.1395
        Ridge - Log Density Deficit          0.4589         0.3170    0.0098
      Extra Trees - Density Deficit          0.5030         0.4126   -0.1897
                        Ridge - Pi1          0.5091         0.4245   -0.2188
    Random Forest - Density Deficit          0.5278         0.4302   -0.3097
Gradient Boosting - Density Deficit          0.5798         0.4095   -0.5808


In [2]:
# ============================================================
# RELATIVE DENSITY — RESIDUAL / ERROR CALIBRATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1. EXPERIMENTAL DATA
# ============================================================

data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "RelativeDensity_pct": [
        99.30, 99.80, 99.80, 99.90, 98.20,
        99.20, 99.10, 99.10, 99.50, 99.90,
        99.95, 99.70, 99.20, 99.70, 98.80,
        99.70, 99.95, 99.40, 99.10, 99.80
    ]
})


# ============================================================
# 2. REMOVE DUPLICATE PROCESS CONDITIONS
# ============================================================

process_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    data
    .groupby(process_cols, as_index=False)
    ["RelativeDensity_pct"]
    .mean()
)

print("Unique experimental conditions:", len(data))


# ============================================================
# 3. PHYSICS FEATURE: Pi1
# ============================================================

Cp = 436.0
k_mm = 0.0103

data["Pi1"] = (
    Cp * data["P_W"]
    /
    (
        k_mm
        * data["v_mm_per_s"]**2
        * data["h_mm"]
    )
)


# ============================================================
# 4. DENSITY DEFICIT
# ============================================================

density = (
    data["RelativeDensity_pct"] / 100.0
)

data["DensityDeficit"] = (
    1.0 - density
)


# ============================================================
# 5. FEATURES
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

X = data[features]

y = data["DensityDeficit"].values


# ============================================================
# 6. WINNING RIDGE MODEL
# ============================================================

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ridge",
        Ridge(alpha=10)
    )
])


# ============================================================
# 7. LOOCV PREDICTIONS
# ============================================================

loo = LeaveOneOut()

predicted_deficit = cross_val_predict(
    model,
    X,
    y,
    cv=loo
)


# ============================================================
# 8. CONVERT BACK TO RELATIVE DENSITY
# ============================================================

predicted_density = (
    1.0 - predicted_deficit
) * 100.0

actual_density = (
    1.0 - y
) * 100.0


# ============================================================
# 9. RESIDUALS
# ============================================================

residual = (
    actual_density
    - predicted_density
)

data["PredictedDensity_pct"] = predicted_density

data["Residual_pct"] = residual

data["AbsoluteError_pct"] = np.abs(
    residual
)


# ============================================================
# 10. BASIC METRICS
# ============================================================

rmse = np.sqrt(
    mean_squared_error(
        actual_density,
        predicted_density
    )
)

mae = mean_absolute_error(
    actual_density,
    predicted_density
)

r2 = r2_score(
    actual_density,
    predicted_density
)


print("\n")
print("=" * 70)
print("DENSITY LOOCV PERFORMANCE")
print("=" * 70)

print(f"RMSE = {rmse:.4f} percentage points")
print(f"MAE  = {mae:.4f} percentage points")
print(f"R²   = {r2:.4f}")


# ============================================================
# 11. RESIDUAL STATISTICS
# ============================================================

print("\n")
print("=" * 70)
print("RESIDUAL STATISTICS")
print("=" * 70)

print(
    f"Mean residual       = {residual.mean():.4f} %"
)

print(
    f"Residual std        = {residual.std(ddof=1):.4f} %"
)

print(
    f"Residual median     = {np.median(residual):.4f} %"
)

print(
    f"Minimum residual    = {residual.min():.4f} %"
)

print(
    f"Maximum residual    = {residual.max():.4f} %"
)

print(
    f"95% residual range  = "
    f"{np.percentile(residual, 2.5):.4f} % "
    f"to "
    f"{np.percentile(residual, 97.5):.4f} %"
)


# ============================================================
# 12. FULL LOOCV TABLE
# ============================================================

print("\n")
print("=" * 70)
print("LOOCV RESIDUAL TABLE")
print("=" * 70)

print(
    data[
        [
            "P_W",
            "v_mm_per_s",
            "h_mm",
            "RelativeDensity_pct",
            "PredictedDensity_pct",
            "Residual_pct",
            "Pi1"
        ]
    ]
    .sort_values("Pi1")
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 13. RESIDUAL VS Pi1
# ============================================================

corr_pi1 = np.corrcoef(
    data["Pi1"],
    data["Residual_pct"]
)[0, 1]

print("\n")
print("=" * 70)
print("RESIDUAL DEPENDENCE ON Pi1")
print("=" * 70)

print(
    f"Correlation(residual, Pi1) = {corr_pi1:.4f}"
)


# ============================================================
# 14. ERROR MODEL OPTIONS
# ============================================================

std_error = residual.std(ddof=1)

mad = np.median(
    np.abs(
        residual -
        np.median(residual)
    )
)

robust_sigma = 1.4826 * mad

print("\n")
print("=" * 70)
print("CANDIDATE ERROR SCALES")
print("=" * 70)

print(
    f"Standard deviation = {std_error:.4f} percentage points"
)

print(
    f"Robust sigma       = {robust_sigma:.4f} percentage points"
)

print(
    f"MAE                = {mae:.4f} percentage points"
)


# ============================================================
# 15. BIAS CHECK
# ============================================================

mean_residual = residual.mean()

print("\n")
print("=" * 70)
print("BIAS CHECK")
print("=" * 70)

if abs(mean_residual) < 0.10:

    print(
        "Residual bias is small."
    )

else:

    print(
        "Noticeable residual bias detected."
    )


# ============================================================
# 16. RECOMMENDED ERROR MODEL
# ============================================================

print("\n")
print("=" * 70)
print("RECOMMENDATION")
print("=" * 70)

print(
    f"Recommended starting error scale: "
    f"{std_error:.4f} percentage points"
)

print(
    "\nThis should be used only after checking "
    "the residual table and residual-vs-Pi1 behavior."
)

Unique experimental conditions: 15


DENSITY LOOCV PERFORMANCE
RMSE = 0.4278 percentage points
MAE  = 0.3344 percentage points
R²   = 0.1395


RESIDUAL STATISTICS
Mean residual       = -0.0055 %
Residual std        = 0.4428 %
Residual median     = 0.1593 %
Minimum residual    = -1.1506 %
Maximum residual    = 0.6263 %
95% residual range  = -0.9343 % to 0.5272 %


LOOCV RESIDUAL TABLE
 P_W  v_mm_per_s   h_mm  RelativeDensity_pct  PredictedDensity_pct  Residual_pct     Pi1
 260        1550 0.0850              99.5000               99.2163        0.2837 53.8940
 270        1600 0.0800              98.2000               99.3506       -1.1506 55.8063
 270        1500 0.0900              99.8000               99.5534        0.2466 56.4401
 280        1550 0.0850              99.9500               99.3237        0.6263 58.0397
 250        1500 0.0800              99.1000               99.2405       -0.1405 58.7918
 260        1550 0.0750              99.3000               99.0178        0.282

In [4]:
# ============================================================
# SAVE RELATIVE DENSITY MINI-MODEL
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


# ============================================================
# 1. EXPERIMENTAL DATA
# ============================================================

data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "RelativeDensity_pct": [
        99.30, 99.80, 99.80, 99.90, 98.20,
        99.20, 99.10, 99.10, 99.50, 99.90,
        99.95, 99.70, 99.20, 99.70, 99.95,
        99.40, 99.10, 99.80, 99.70, 99.80
    ]
})


# ============================================================
# 2. REMOVE DUPLICATE PROCESS CONDITIONS
# ============================================================

process_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    data
    .groupby(
        process_cols,
        as_index=False
    )["RelativeDensity_pct"]
    .mean()
)


# ============================================================
# 3. CREATE DENSITY DEFICIT
# ============================================================

density = (
    data["RelativeDensity_pct"] / 100.0
)

data["DensityDeficit"] = (
    1.0 - density
)


# ============================================================
# 4. INPUT FEATURES
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

X = data[features]

y = data["DensityDeficit"].values


# ============================================================
# 5. RIDGE MINI-MODEL
# ============================================================

density_model = Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "ridge",
        Ridge(alpha=10)
    )
])


# ============================================================
# 6. LOOCV RESIDUALS
# ============================================================

loo = LeaveOneOut()

predicted_deficit = cross_val_predict(
    density_model,
    X,
    y,
    cv=loo
)


# Convert predictions back to density %
predicted_density_pct = (
    1.0 - predicted_deficit
) * 100.0

actual_density_pct = (
    data["RelativeDensity_pct"]
    .values
)


# Residual:
# Actual - Predicted
residual = (
    actual_density_pct
    - predicted_density_pct
)


# ============================================================
# 7. ROBUST RESIDUAL POOL
# ============================================================

median_residual = np.median(residual)

mad = np.median(
    np.abs(
        residual - median_residual
    )
)

robust_sigma = 1.4826 * mad


# Keep residuals within ±2 robust sigma
residual_pool = residual[
    np.abs(
        residual - median_residual
    )
    <= 2 * robust_sigma
]


# ============================================================
# 8. FIT FINAL MODEL ON ALL EXPERIMENTAL DATA
# ============================================================

density_model.fit(
    X,
    y
)


# ============================================================
# 9. SAVE MODEL
# ============================================================

joblib.dump(
    density_model,
    "RelativeDensity_mini_model.pkl"
)


# ============================================================
# 10. SAVE RESIDUAL INFORMATION
# ============================================================

density_error_info = {

    "residual_pool_pct":
        residual_pool,

    "all_residuals_pct":
        residual,

    "mean_residual_pct":
        residual.mean(),

    "std_residual_pct":
        residual.std(ddof=1),

    "median_residual_pct":
        median_residual,

    "robust_sigma_pct":
        robust_sigma,

    "features":
        features
}


joblib.dump(
    density_error_info,
    "RelativeDensity_error_model.pkl"
)


# ============================================================
# 11. DISPLAY RESULTS
# ============================================================

print("=" * 70)
print("RELATIVE DENSITY MINI-MODEL SAVED")
print("=" * 70)

print(
    "Model file:"
    " RelativeDensity_mini_model.pkl"
)

print(
    "Error file:"
    " RelativeDensity_error_model.pkl"
)

print(
    f"\nLOOCV residual std = "
    f"{residual.std(ddof=1):.4f} %"
)

print(
    f"Robust sigma       = "
    f"{robust_sigma:.4f} %"
)

print(
    f"Residual pool size = "
    f"{len(residual_pool)}"
)

print("\nResidual pool:")
print(residual_pool)

RELATIVE DENSITY MINI-MODEL SAVED
Model file: RelativeDensity_mini_model.pkl
Error file: RelativeDensity_error_model.pkl

LOOCV residual std = 0.4700 %
Robust sigma       = 0.2953 %
Residual pool size = 14

Residual pool:
[-0.11533827 -0.22114105 -0.35821677  0.1382864   0.24486976  0.28448751
 -0.31808217  0.15952381  0.27185685  0.19913123 -0.05977403  0.61947346
  0.48239774 -0.06089576]
